<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/probable-journey/blob/main/01.%20Status-cleanser/Status_Cleanser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Installing OpenAI
!pip install openai

In [6]:
#File upload
from google.colab import files
uploaded = files.upload()

Saving messy_claims.csv to messy_claims.csv


In [7]:
#Printing the file path
import os
print(os.listdir())

['.config', 'sample_data', 'messy_claims.csv']


In [8]:
#Storing the OpenAI key securely in colab
from getpass import getpass

OPENAI_API_KEY = getpass(
    "Enter your OpenAI API key: "
)

Enter your OpenAI API key: ··········


In [9]:
#Getting the status by rules/AI
import pandas as pd
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI client created successfully")

CANONICAL_STATUSES = [
    "approved",
    "pending",
    "rejected"
]

RULES = {
    "approved": "approved",
    "appr": "approved",

    "pending review": "pending",
    "pending_review": "pending",
    "in review": "pending",
}

RULES = {
    "approved": "approved",
    "appr": "approved",
    "pending review": "pending",
    "pending_review": "pending",
    "in review": "pending",
}

## Clean by Rules function
def clean_with_rules(raw_value: str):

    normalized = (
        raw_value
        .strip()
        .lower()
        .replace("_", " ")
    )
    return RULES.get(normalized)

#Cleaning with AI function
def clean_with_ai(raw_value: str) -> str:

    print(
        f"[AI fallback] "
        f"Classifying: '{raw_value}'"
    )

    try:

        response = client.responses.create(
            model="gpt-5-mini",
            input=[

                {
                    "role": "developer",

                    "content": (
                        "You classify claim statuses. "
                        "You must return exactly one "
                        "of these three words: "
                        "approved, pending, rejected. "
                        "Do not explain your answer. "
                        "Return only one word."
                    )
                },

                {
                    "role": "user",
                    "content": (
                        f"Classify this claim status: "
                        f"{raw_value}"
                    )
                }
            ]
        )

        result = (
            response.output_text
            .strip()
            .lower()
        )


        # Validate AI response
        if result not in CANONICAL_STATUSES:
            print(
                f"WARNING: "
                f"Unexpected AI response: {result}"
            )

            return "pending"
        return result

    except Exception as error:
        print(
            f"AI Error: {error}"
        )
        return "pending"


OpenAI client created successfully


In [10]:
def process_file(
    input_csv: str,
    output_csv: str
):

    print("Reading CSV file...")

    df = pd.read_csv(
        input_csv
    )

    cleaned_statuses = []
    methods_used = []

    for raw_value in df["status"]:

        # Convert value to string
        raw_value = str(raw_value)

        # Try rules first
        rule_result = clean_with_rules(
            raw_value
        )

        if rule_result:
            cleaned_statuses.append(
                rule_result
            )
            methods_used.append(
                "rule"
            )

        else:
            ai_result = clean_with_ai(
                raw_value
            )
            cleaned_statuses.append(
                ai_result
            )
            methods_used.append(
                "ai"
            )


    # Add cleaned status column
    df["status_cleaned"] = (
        cleaned_statuses
    )

    # Add method column
    df["method_used"] = (
        methods_used
    )

    # Save file
    df.to_csv(
        output_csv,
        index=False
    )

    # Count methods
    rule_count = methods_used.count(
        "rule"
    )
    ai_count = methods_used.count(
        "ai"
    )
    print("\nPROCESS COMPLETE")
    print(
        f"Rule matches: {rule_count}"
    )
    print(
        f"AI classifications: {ai_count}"
    )
    print(
        f"Saved file: {output_csv}"
    )

In [11]:
process_file(
    "messy_claims.csv",
    "cleaned_claims.csv"
)

Reading CSV file...
[AI fallback] Classifying: 'accepted'
AI Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
[AI fallback] Classifying: 'waiting'
AI Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
[AI fallback] Classifying: 'denied'
AI Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
[AI fallback] Classifying: 'Claim Declined'
AI Error: Error code: 42